# Analysis of segmented FLIM data 

For evey cell with phasor coordinates $(G,S)_{cell}$ and projection on the metabolic trajectory $(G,S)_{projection}$, the Metabolic Index is calculated as the fraction f that satisfies the following equation:
$$
(G,S)_{projection} = (1-f)(G_{free},S_{free}) + f(G_{bound},S_{bound})
$$
$$
\tau_{free} = 0.4 \space(ns)
$$
$$
\tau_{bound} \in [3.0,3.4] \space(ns)
$$

At the same time, the phasor-derived fast lifetime is calculated as:
$$
\tau_{fast}=\frac{S}{\omega\cdot G}
$$


## Load libs & configure SAVE_DIR


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.patches as mpatches
from scipy.optimize import curve_fit
from statannotations.Annotator import Annotator
import os
from tqdm import tqdm
from datetime import datetime
from aicsimageio import AICSImage
from skimage.measure import regionprops
import warnings
import logging

# Suppress all warnings and logging messages
warnings.filterwarnings('ignore')

# Set environment variables to suppress bfio and Maven download messages
os.environ['BFIO_LOG_LEVEL'] = 'ERROR'
os.environ['MAVEN_OPTS'] = '-q'  # Quiet Maven output
os.environ['JAVA_TOOL_OPTIONS'] = '-Dlog4j2.configurationFile=SYSTEM_OUT'

# Suppress specific loggers
logging.getLogger('bfio').setLevel(logging.ERROR)
logging.getLogger('aicsimageio').setLevel(logging.ERROR)
logging.getLogger('cjdk').setLevel(logging.ERROR)
logging.getLogger('jpype').setLevel(logging.ERROR)

# Set console logging to ERROR level to suppress INFO messages
logging.basicConfig(level=logging.ERROR)

# Additional suppression for Maven/Java output
import subprocess
import sys
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO

# Redirect stdout/stderr temporarily during imports
f = StringIO()
with redirect_stdout(f), redirect_stderr(f):
    from aicsimageio import AICSImage


import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
from matplotlib import pyplot as plt
%matplotlib inline



## Configure **SAVE_DIR** and **DATA_DIR** and other global variables

In [ ]:
SAVE_DIR = # Set your desired save directory here
IMG_ROOT_DIR = # Set your image root directory here
MASK_DIR = # Set your mask directory here
STANDARDS_DIR = # Set your FLIM standards directory here


In [ ]:
today = datetime.today()
date_str = today.strftime("%d-%m-%Y")
SAVE_DIR = os.path.join(SAVE_DIR, f"Analysis_{date_str}")
os.makedirs(SAVE_DIR, exist_ok=True)
print("Results will be saved to:", SAVE_DIR)

In [ ]:
MAX_H = 25 # Maximum width of cell gallery tiles 
MAX_W = 25 # Maximum height of cell gallery tiles 
PIXEL_SIZE_UM = 0.86664795  # Microns per pixel
OMEGA = 2 * np.pi * 80e6  # Angular frequency for 80 MHz in rad/s
TAU_FREE = 0.4e-9  # Free NAD(P)H lifetime in seconds
TAU_BOUND = 3.1e-9  # Bound NAD(P)H lifetime in seconds
FIT_TAU_BOUND = True  # Whether to fit for bound NAD(P)H lifetime or use the fixed value above

## Functions

In [ ]:
def get_phasor_offsets(row):
    """ Calculate phasor offsets based on background G and S components. Get data from the image and masks linked to the row. """
    img_path = row['image_path']
    mask_path = row['mask_path']
    
    # Load image and mask
    img = AICSImage(img_path)
    mask = AICSImage(mask_path)
    
    # Extract G and S channels from the image
    g_channel = img.get_image_data("YX", C=5)  # Assuming G is channel 5
    s_channel = img.get_image_data("YX", C=6)  # Assuming S is channel 6
    
    # Extract background region from the mask
    background_mask = (mask.get_image_data("YX") == 0)  # Assuming background is labeled as 0
    
    # Calculate mean G and S in the background region
    g_background = np.mean(g_channel[background_mask])
    s_background = np.mean(s_channel[background_mask])
    
    return g_background, s_background

In [ ]:
def apply_phasor_correction(row, g_channel='G', s_channel='S'):
    """ Apply phasor correction to G and S channels """
    
    g_offset = row['g_off0et']
    s_offset = row['s_offset']
    g = row[g_channel]
    s = row[s_channel]
    g_corrected = g - g_offset
    s_corrected = s - s_offset
    r = 2**16-1  # Let this be 16bit conversion
    g_norm = g_corrected / r
    s_norm = s_corrected / (2*r) # S is halved in phasor calculation
    
    
    return g_norm, s_norm

In [ ]:
def scale_phasor(row, g_channel='G', s_channel='S'):
    """ Scale phasor coordinates assuming a 16 bit depth expansion """
    
    g = row[g_channel]
    s = row[s_channel]
    
    g_scaled = g / (2**16 - 1)
    s_scaled = s / (2*(2**16 - 1))
    
    return g_scaled, s_scaled

In [ ]:
def construct_paths(row):
    img_root_dir = IMG_ROOT_DIR 
    mask_dir = MASK_DIR

    file = row['file']
    #string join with '_'
    parts = file.split('_')
    img_dir = os.path.join(img_root_dir, '_'.join(parts[:2]))
    img_path = os.path.join(img_dir, file)
    mask_file = 'masks-30_' + file.replace('.ome.tif', '') + '_Image_0.ome.tif'
    mask_path = os.path.join(mask_dir, mask_file)
    return pd.Series([img_path, mask_path])

In [ ]:
def save_gallery_2(
    df,
    indexes,
    SAVE_DIR,
    label=None,
    lifetime=True,
    photons=True,
    auto_contrast=False,
    n_cols=5,
    add_colorbar=True,
    cmap="viridis",
    feature="area",
    size=2.5,
    # optional caps; set None for fully adaptive
    max_h=None,
    max_w=None,
    pixel_size_um=None,
    scalebar_um=10,
    scalebar_color="white",
    scalebar_pad_px=1,
    scalebar_lw=3,
    scalebar_fontsize=7,
    scalebar_margin_px=10,   # <-- NEW: reserved strip at bottom
    vmin=0.5,
    vmax=2.5,
    dpi=300,
    interpolation="nearest",
    verbose=True,
):
    """
    Save a gallery of cell crops with an accurate scalebar placed in a dedicated
    bottom margin strip (never overlaps the cell).

    Tiles adapt to each crop size (unless max_h/max_w are provided as caps).
    Scale bar is drawn in pixel data coordinates (accurate regardless of figure size).
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize
    import seaborn as sns
    from aicsimageio import AICSImage
    from datetime import datetime

    today = datetime.today()

    # -------------------------
    # Early exits / sanity
    # -------------------------
    if indexes is None:
        indexes = []
    indexes = list(indexes)
    n = len(indexes)
    if n == 0:
        if verbose:
            print("save_gallery_2: no indexes provided, nothing to plot.")
        return None

    if not (photons or lifetime):
        raise ValueError("save_gallery_2: at least one of photons/lifetime must be True.")

    scalebar_margin_px = int(max(0, scalebar_margin_px))

    # -------------------------
    # Colormap resolution
    # -------------------------
    def _resolve_cmap(c):
        if isinstance(c, str):
            try:
                if c in [
                    "viridis", "plasma", "inferno", "magma", "cividis",
                    "rocket", "mako", "flare", "crest"
                ] or c.startswith(("ch:", "blend:")):
                    return sns.color_palette(c, as_cmap=True)
                return cm.get_cmap(c)
            except Exception:
                if verbose:
                    print(f"Warning: Colormap '{c}' not found. Using 'viridis'.")
                return cm.get_cmap("viridis")
        if hasattr(c, "__call__"):
            return c
        try:
            return sns.blend_palette(c, as_cmap=True)
        except Exception:
            if verbose:
                print(f"Warning: Could not create colormap from {c}. Using 'viridis'.")
            return cm.get_cmap("viridis")

    colormap = _resolve_cmap(cmap)
    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)

    # -------------------------
    # Helpers
    # -------------------------
    def _clamp_bbox(minr, minc, maxr, maxc, H, W):
        minr = int(np.floor(minr))
        minc = int(np.floor(minc))
        maxr = int(np.ceil(maxr))
        maxc = int(np.ceil(maxc))

        minr = max(0, minr)
        minc = max(0, minc)
        maxr = min(H, maxr)
        maxc = min(W, maxc)

        h = maxr - minr
        w = maxc - minc
        return minr, minc, maxr, maxc, h, w

    def _safe_uint8_contrast(img2d, mask=None):
        x = img2d.astype(np.float32)

        if mask is not None and getattr(mask, "any", lambda: False)():
            vals = x[mask]
        else:
            vals = x.ravel()

        if vals.size == 0:
            return np.zeros_like(img2d, dtype=np.uint8)
        # Using 0.8 and 1.2 scaling factors to avoid extreme clipping
        if auto_contrast:
            lo = np.percentile(vals, 1)*0.8
            hi = np.percentile(vals, 99)*1.2
        else:
            lo = np.min(vals)*0.8
            hi = np.max(vals)*1.2

        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return np.zeros_like(img2d, dtype=np.uint8)

        y = (x - lo) / (hi - lo)
        y = np.clip(y, 0, 1)
        return (y * 255).astype(np.uint8)

    def _imshow_tile(ax, rgb, tile_w_px, tile_h_px_total):
        # Pixel coordinate system: x in [0, tile_w], y in [0, tile_h_total]
        ax.imshow(
            rgb,
            extent=[0, tile_w_px, tile_h_px_total, 0],
            interpolation=interpolation,
        )
        ax.set_xlim(0, tile_w_px)
        ax.set_ylim(tile_h_px_total, 0)
        ax.axis("off")
        ax.set_facecolor("black")

    def _add_scalebar_in_strip(ax, tile_w_px, img_h_px, strip_h_px):
        """
        Draw scalebar in the bottom strip region.
        Coordinates are pixel data coords with extent=[0,w,total_h,0].
        Strip occupies y in [img_h_px, img_h_px + strip_h_px] (top->bottom).
        """
        if strip_h_px <= 0:
            return
        if pixel_size_um is None or scalebar_um is None:
            return
        if pixel_size_um <= 0 or scalebar_um <= 0:
            return

        desired_bar_px = int(round(scalebar_um / pixel_size_um))
        if desired_bar_px < 2:
            return

        # padding should be sensible relative to strip height
        pad_x = int(min(scalebar_pad_px, max(3, round(0.04 * tile_w_px))))
        pad_y = int(min(scalebar_pad_px, max(2, round(0.25 * strip_h_px))))

        # Fit bar into width; if it doesn't fit, shorten and label actual length
        max_bar_px = max(2, tile_w_px - 2 * pad_x)
        bar_px = min(desired_bar_px, max_bar_px)
        bar_um = bar_px * pixel_size_um

        x1 = tile_w_px - pad_x
        x0 = x1 - bar_px

        # Place bar vertically centered within strip
        y_strip_top = img_h_px
        y_strip_bottom = img_h_px + strip_h_px
        y0 = int(round((y_strip_top + y_strip_bottom) / 2))

        ax.plot([x0, x1], [y0, y0],
                color=scalebar_color, lw=scalebar_lw, solid_capstyle="butt")

        # Label slightly above the bar, but still inside strip
        y_text = max(y_strip_top + 2, y0 - pad_y)
        ax.text((x0 + x1) / 2, y_text,
                f"{bar_um:.0f} µm" if bar_um >= 1 else f"{bar_um:.1f} µm",
                color=scalebar_color,
                ha="center", va="bottom",
                fontsize=scalebar_fontsize)
    def _get_lifetime(img_crop, omega=OMEGA):
        """ Compute lifetime from G and S channels. """
        G = img_crop[5].astype(np.float32)
        S = img_crop[6].astype(np.float32)
        # Check if G and S are valid
        if G is None or S is None:
            return np.zeros_like(img_crop[0], dtype=np.float32)
        if np.all(G == 0):
            return np.zeros_like(img_crop[0], dtype=np.float32)
        denom = G * G + S * S
        denom[denom == 0] = 1e-6

        tau = (1e9 / omega) * (S / G) # in ns
        tau = tau * (denom / denom)  # maintain shape

        return tau
    # -------------------------
    # Layout
    # -------------------------
    n_rows = int(np.ceil(n / n_cols))
    plt.close("all")

    fig_width = n_cols * size * (1.2 if (add_colorbar and lifetime) else 1.0)
    fig_height = n_rows * size
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
    fig.patch.set_facecolor("black")

    if add_colorbar and lifetime:
        fig.subplots_adjust(wspace=0.05, hspace=0.05, right=0.85)
    else:
        fig.subplots_adjust(wspace=0.05, hspace=0.05)

    axes = np.array(axes).reshape(-1)

    for ax in axes[n:]:
        ax.axis("off")

    colorbar_mappable = None

    # -------------------------
    # Main loop
    # -------------------------
    for ax, idx in zip(axes[:n], indexes):
        try:
            row = df.loc[idx]
        except Exception as e:
            if verbose:
                print(f"Index {idx} not found in df: {e}")
            ax.axis("off")
            continue

        img_path = row.get("img_path", None)
        if not img_path or not os.path.exists(img_path):
            if verbose:
                print(f"Missing img_path for idx={idx}: {img_path}")
            ax.axis("off")
            continue

        try:
            image = AICSImage(img_path)
            img = image.get_image_data("CYX")
        except Exception as e:
            if verbose:
                print(f"Failed reading image for idx={idx}: {e}")
            ax.axis("off")
            continue

        H_img, W_img = img.shape[1], img.shape[2]

        try:
            minr = row["minr"]; minc = row["minc"]; maxr = row["maxr"]; maxc = row["maxc"]
        except Exception as e:
            if verbose:
                print(f"Missing bbox columns for idx={idx}: {e}")
            ax.axis("off")
            continue

        minr, minc, maxr, maxc, h, w = _clamp_bbox(minr, minc, maxr, maxc, H_img, W_img)
        if h <= 1 or w <= 1:
            if verbose:
                print(f"Skipping idx={idx} due to invalid crop size h={h}, w={w}")
            ax.axis("off")
            continue

        img_crop = img[:, minr:maxr, minc:maxc]

        # Mask
        cell_mask_crop = np.ones((h, w), dtype=bool)
        mask_path = row.get("mask_path", None)
        cell_label = row.get("label", None)

        if mask_path and os.path.exists(mask_path) and cell_label is not None:
            try:
                mask_img = AICSImage(mask_path).get_image_data("YX")
                mh, mw = mask_img.shape
                minr_m, minc_m, maxr_m, maxc_m, hh, ww = _clamp_bbox(minr, minc, maxr, maxc, mh, mw)
                if hh == h and ww == w:
                    cell_mask_crop = (mask_img[minr_m:maxr_m, minc_m:maxc_m] == cell_label)
                else:
                    if verbose:
                        print(f"Mask crop mismatch for idx={idx}; using all-True mask.")
            except Exception as e:
                if verbose:
                    print(f"Failed reading/using mask for idx={idx}: {e}. Using all-True mask.")

        # Optional cap (clips crop, still accurate scalebar)
        if max_h is not None and h > max_h:
            img_crop = img_crop[:, :max_h, :]
            cell_mask_crop = cell_mask_crop[:max_h, :]
            h = max_h
        if max_w is not None and w > max_w:
            img_crop = img_crop[:, :, :max_w]
            cell_mask_crop = cell_mask_crop[:, :max_w]
            w = max_w

        # Tile geometry
        img_h = h
        strip_h = scalebar_margin_px
        tile_h_total = img_h + strip_h
        tile_w = w * (2 if (photons and lifetime) else 1)

        # Full tile canvas including strip (strip stays black)
        tile = np.zeros((tile_h_total, tile_w, 3), dtype=np.uint8)

        # Photons in left image area
        if photons and img_crop.shape[0] >= 1:
            photons_2d = img_crop[0]
            photons_u8 = _safe_uint8_contrast(photons_2d, mask=cell_mask_crop)
            ph = photons_u8.copy()
            ph[~cell_mask_crop] = (ph[~cell_mask_crop] // 4).astype(np.uint8)
            tile[:img_h, :w, :] = np.stack([ph, ph, ph], axis=-1)

        # Lifetime in right image area (or left if photons False)
        if lifetime and img_crop.shape[0] >= 2:
            life = _get_lifetime(img_crop) #img_crop[1].astype(np.float32).copy()
            life[~cell_mask_crop] *= 0.25
            life_rgb = (colormap(norm(life))[:, :, :3] * 255).astype(np.uint8)

            x_off = (w if (photons and lifetime) else 0)
            tile[:img_h, x_off:x_off + w, :] = life_rgb

            if colorbar_mappable is None and add_colorbar:
                colorbar_mappable = cm.ScalarMappable(norm=norm, cmap=colormap)
                colorbar_mappable.set_array([])

        # Display tile
        _imshow_tile(ax, tile, tile_w, tile_h_total)

        # Scalebar in strip
        _add_scalebar_in_strip(ax, tile_w_px=tile_w, img_h_px=img_h, strip_h_px=strip_h)

    # -------------------------
    # Colorbar
    # -------------------------
    if add_colorbar and lifetime and (colorbar_mappable is not None):
        cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])
        cbar = fig.colorbar(colorbar_mappable, cax=cbar_ax)
        cbar.set_label("Lifetime (ns)", rotation=270, labelpad=15, color="white")
        cbar.ax.yaxis.set_tick_params(color="white")
        cbar.ax.yaxis.label.set_color("white")
        plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")

    # -------------------------
    # Save
    # -------------------------
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path = os.path.join(
        SAVE_DIR,
        f"{feature}_cell_gallery_{label}_{today.strftime('%d%m%Y')}.png",
    )
    plt.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.1,
        facecolor="black",
        edgecolor="none",
    )
    plt.show()

    return save_path


In [ ]:
def plot_representative_cells(
    data,
    savedir=SAVE_DIR,
    n_cells=20,
    lifetime=True,
    photons=True,
    auto_contrast=False,
    label=None,
    add_colorbar=True,
    feature='area',
    area_feature='area',
    ncols=5,
    pixel_size_um=PIXEL_SIZE_UM,
    scalebar_um=10,
    scalebar_color='white',
    scalebar_pad_px=1
):
    """ Plot representative cells by sampling from feature distribution near median value of the feature. """
    # Get median feature value
    median_feature = data[feature].median()
    # Get cells within 10% of median feature
    feature_tolerance = 0.1 * median_feature
    candidate_cells = data[(data[feature] >= (median_feature - feature_tolerance)) &
                           (data[feature] <= (median_feature + feature_tolerance))]
    # Within these, further filter to cells with area within 1 std of mean area
    area_std = data[area_feature].std()
    area_mean = data[area_feature].mean()
    candidate_cells = candidate_cells[(candidate_cells[area_feature] >= (area_mean - area_std)) &
                                      (candidate_cells[area_feature] <= (area_mean + area_std))]

    print(f"Found {len(candidate_cells)} candidate cells near median {feature} of {median_feature:.2f}.")

    # Sample n_cells from candidate cells
    if len(candidate_cells) < n_cells:
        print(f"Warning: Only {len(candidate_cells)} candidate cells found, less than requested {n_cells}. Using all candidates.")
        sampled_cells = candidate_cells
    else:
        sampled_cells = candidate_cells.sample(n=n_cells, random_state=42)

    # Save gallery
    save_gallery_2(
        data,
        sampled_cells.index.tolist(),
        savedir,
        label=label,
        lifetime=lifetime,
        photons=photons,
        auto_contrast=auto_contrast,
        n_cols=ncols,
        add_colorbar=add_colorbar,
        feature=feature,
        pixel_size_um=pixel_size_um,
        scalebar_um=scalebar_um,
        scalebar_color=scalebar_color,
        scalebar_pad_px=scalebar_pad_px
    )

## Load data

In [ ]:
data_dir = MASK_DIR
files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
print(f"Found {len(files)} files for analysis.")
data_list = []
for file in tqdm(files):
    filepath = os.path.join(data_dir, file)
    df = pd.read_csv(filepath)
    df['Source file'] = file
    df['mouse_id'] = file.split('_')[1]
    df['condition'] = file.split('_')[1][:-1]
    df['treatment'] = file.split('_')[2]
    df['replicate'] = file.split('_')[3]
    data_list.append(df)
data = pd.concat(data_list, ignore_index=True)
print("Combined data shape:", data.shape)



In [ ]:
# Annotate intensity-0 - intensity-6 columns
channels_map = {
    'intensity-0': 'photon_counts',
    'intensity-1': 'fast_lifetime',
    'intensity-2': 'phasor_mask_red',
    'intensity-3': 'phasor_mask_green',
    'intensity-4': 'phasor_mask_blue',
    'intensity-5': 'phasor_g_component',
    'intensity-6': 'phasor_s_component'
}
for key, value in channels_map.items():
    for prefix in ['min_', 'mean_', 'max_']:
        col_name = f"{prefix}{key}"
        if col_name in data.columns:
            new_col_name = f"{prefix}{value}"
            data.rename(columns={col_name: new_col_name}, inplace=True)


In [ ]:
keep_cols = ['mouse_id', 'condition', 'treatment', 'replicate', 'label',
            'frame', 'file', 'scene',  'x_voxel_size', 'y_voxel_size', 'physical_units_applied',
            'area', 'bbox-0', 'bbox-1', 'bbox-2', 'bbox-3',
            'major_axis_length', 'minor_axis_length', 'eccentricity', 'solidity',
            'min_photon_counts', 'mean_photon_counts', 'max_photon_counts',
            'min_fast_lifetime', 'mean_fast_lifetime', 'max_fast_lifetime',
            'min_phasor_g_component', 'mean_phasor_g_component', 'max_phasor_g_component',
            'min_phasor_s_component', 'mean_phasor_s_component', 'max_phasor_s_component']
data = data[keep_cols]

In [ ]:
data['fast_tau_ns'] = data['mean_fast_lifetime'] / 1000  # Convert ps to ns

In [ ]:
# Replace LC to WT
data['condition'] = data['condition'].replace({'LC': 'WT'})

In [ ]:
data['folder'] = data['file'].apply(lambda x: '_'.join(x.split('_')[:2]))

## Load calibration standard

In [ ]:
def get_affine_transform_params(std_tau_mean, std_g_mean, std_s_mean, gray_scaling=0.001, laser_freq=80e6):
    """ Calculate offset and scaling factors for affine transformation based on standard values

        Args:
            std_tau_mean (float): Mean lifetime of the standard in gray AU per ns
            std_g_mean (float): Mean G component of the standard in LASX AU
            std_s_mean (float): Mean S component of the standard in LASX AU
            gray_scaling (float): Scaling factor for gray values (ns/gray AU)
            laser_freq (float): Laser repetition frequency in Hz ( default 80e6 Hz)
        Returns:
            offset (float): Offset to be applied to G and S components
            scaling (float): Scaling factor to be applied to G and S components
            g (float): Calculated G component for the standard
            s (float): Calculated S component for the standard
    """
    tau_seconds = std_tau_mean * gray_scaling * 1e-9  # Convert to seconds
    omega = 2 * np.pi * laser_freq  # Angular frequency for given laser frequency
    g = 1 / (1 + (tau_seconds * omega)**2)
    s = (tau_seconds * omega) / (1 + (tau_seconds * omega)**2)
    scaling = (std_g_mean-std_s_mean)/(g - s)
    offset = std_g_mean - scaling * g
    offset_s = std_s_mean - scaling * s
    assert np.isclose(offset, offset_s), "Offsets for G and S do not match!"
    return offset, scaling, g, s

In [ ]:
standard_dir = STANDARDS_DIR
standard_files = [f for f in os.listdir(standard_dir) if f.endswith('.ome.tif')]

In [ ]:
# Load all standards and calculate mean G, S, Tau, and offsets and save in dataframe
#standards_df = pd.DataFrame(columns=['file', 'std_g_mean', 'std_s_mean', 'std_tau_mean','std_g', 'std_s', 'offset', 'scaling'])

standards = []
for standard_file in tqdm(standard_files):
    standard_path = os.path.join(standard_dir, standard_file)
    standard_img = AICSImage(standard_path)
    # Extract date from file metadata
    metadata = standard_img.metadata.__dict__
    date = metadata['images'][0].acquisition_date
    standard_counts = standard_img.get_image_data("YX", C=0)
    standard_mean_counts = np.mean(standard_counts)
    total_counts = np.sum(standard_counts)
    # Calculate weighted mean G, S, Tau
    standard_g_channel_mean = np.average(standard_img.get_image_data("YX", C=5), weights=standard_counts)
    standard_s_channel_mean = np.average(standard_img.get_image_data("YX", C=6), weights=standard_counts)
    standard_tau_channel_mean = np.average(standard_img.get_image_data("YX", C=1), weights=standard_counts)
    offset, scaling, standard_g, standard_s = get_affine_transform_params(
        std_tau_mean=standard_tau_channel_mean,
        std_g_mean=standard_g_channel_mean,
        std_s_mean=standard_s_channel_mean,
        gray_scaling=0.001,
        laser_freq=80e6
    )
    # Store results
    standard ={
        'date': date.strftime("%d-%m-%Y"),
        'file': standard_file,
        'total_counts': total_counts,
        'std_mean_counts': standard_mean_counts,
        'std_g_mean': standard_g_channel_mean,
        'std_s_mean': standard_s_channel_mean,
        'std_tau_mean': standard_tau_channel_mean,
        'std_g': standard_g,
        'std_s': standard_s,
        'offset': offset,
        'scaling': scaling
    }
    standard_df = pd.DataFrame(standard, index=[0])
    standards.append(standard_df)
  
    # Calculate and plot g and s with applied offset and scaling
    g_corrected = (standard_g_channel_mean - offset) / scaling
    s_corrected = (standard_s_channel_mean - offset) / scaling
    #plt.scatter(g_corrected, s_corrected, marker='x', s=50, label=f"{standard_file.split('_')[0]} (corrected)")
standards_df = pd.concat(standards, ignore_index=True)

    


In [ ]:
# Plot standard in the phasor plot
plt.figure(figsize=(8,4))
# Add universal semicircle
theta = np.linspace(0, np.pi/2, 100)
u = np.cos(theta)**2
v = np.sin(theta)*np.cos(theta)
plt.plot(u, v, color='black', linestyle='--', label='Universal Semicircle')
plt.xlim(0, 1)
plt.ylim(0, 0.5)
plt.xlabel('G')
plt.ylabel('S')
#plt.title('Phasor Plot with Standard Coumarin-6')
order_map = {'16-12-2023': 1, '27-01-2024': 2, '03-02-2024': 3}
standards_df['order'] = standards_df['date'].map(order_map)
standards_df.sort_values('order', inplace=True)
for _, standard in standards_df.iterrows():
    standard_g = (standard['std_g_mean'] - standard['offset']) / standard['scaling']
    standard_s = (standard['std_s_mean'] - standard['offset']) / standard['scaling']
    # Plot standard point
    plt.scatter(standard_g, standard_s, s=50, label=f"{standard['date']} measurement", marker='o', edgecolor='black', alpha=0.9)
plt.legend(loc='upper right', fontsize='small', bbox_to_anchor=(1.4, 1))

# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Phasor_plot_standards_coumarin6.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Select the offset and scale of the COUMARIN-6 measurement with the highest total counts 
standard_to_use = standards_df.loc[standards_df['total_counts'].idxmax()]
offset = standard_to_use['offset']
scaling = standard_to_use['scaling']
print(f"Using standard from {standard_to_use['date']} with offset {offset} and scaling {scaling}")
data['g_corrected'] = (data['mean_phasor_g_component'] - offset) / scaling
data['s_corrected'] = (data['mean_phasor_s_component'] - offset) / scaling


## Recompute cell features 
Done in order to adjust g and s according to standard 

In [ ]:
# Apply custom function with skimage.measure.regionprops to do intensity scale on lifetime channel when averaging per cell
# Create dataframe with all files, then generate image and mask paths, then apply function to get intensity scaled lifetime per cell
data_paths = pd.DataFrame(columns=['file', 'image_path', 'mask_path'])
data_paths['file'] = data['file'].unique()
data_paths[['image_path', 'mask_path']] = data_paths.apply(construct_paths, axis=1)

In [ ]:
# Initialize results storage
intensity_scaled_results = []

for idx, row in tqdm(data_paths.iterrows(), total=data_paths.shape[0]):
    try:
        img_path = row['image_path']
        mask_path = row['mask_path']
        
        # Load image and mask
        img = AICSImage(img_path)
        x_pixel_size = img.physical_pixel_sizes.X
        y_pixel_size = img.physical_pixel_sizes.Y
        spacing = (y_pixel_size, x_pixel_size)
        mask = AICSImage(mask_path)
        
        # Get image data and check dimensions
        img_data = img.get_image_data("YXC")  # Shape should be (Y, X, C)
        #print(f"Image shape: {img_data.shape}")  # Debug print
        
        # Keep only photon counts (channel 0), lifetime (channel 1), g coord (channel 5), s coord (channel 6)
        if img_data.shape[2] >= 7:
            img_data = img_data[..., [0, 1, 5, 6]]  # Keep only channels 0, 1, 5, 6
        else:
            print(f"Warning: Image has only {img_data.shape[2]} channels")
            continue
            
        # Get labels from the mask
        labels = mask.get_image_data("YX")
        
        regions = regionprops(labels, spacing=spacing)

        weighted_means = []

        
        for r in regions:
            try:
                rr, cc = r.coords.T

                ch0 = img_data[rr, cc, 0]   # weights
                ch1 = img_data[rr, cc, 1]   # values
                ch5 = img_data[rr, cc, 2]   # g coord
                ch6 = img_data[rr, cc, 3]   # s coord

                counts_mean = np.mean(ch0)
                counts_median = np.median(ch0)
                tau_wmean = np.average(ch1, weights=ch0)* 1e-3  # Convert ps to ns
                g_wmean = np.average(ch5, weights=ch0)
                s_wmean = np.average(ch6, weights=ch0)
                total_counts = np.sum(ch0)
                # Corrected g and s
                g_corrected = (g_wmean - offset) / scaling
                s_corrected = (s_wmean - offset) / scaling
                # Get bounding box
                minr, minc, maxr, maxc = r.bbox
                # Get centroid
                centroid_r, centroid_c = r.centroid
                weighted_means.append({'label': r.label,
                                    'area' : r.area,
                                    'centroid_r': centroid_r,
                                    'centroid_c': centroid_c,
                                    'minr': minr,
                                    'minc': minc,
                                    'maxr': maxr,
                                    'maxc': maxc,
                                    'eccentricity': r.eccentricity,
                                    'solidity': r.solidity,
                                    'tau': tau_wmean,
                                    'g_corrected': g_corrected,
                                    's_corrected': s_corrected,
                                    'mean_photon_counts': counts_mean,
                                    'median_photon_counts': counts_median,
                                    'total_photon_counts': total_counts})
            except Exception as e:
                print(f"Error processing region {r.label} in file {row['file']}: {e}. Skipping this region.")
                continue
        # Create DataFrame for this image
        df_image = pd.DataFrame(weighted_means)
        # Add metadata columns
        df_image['file'] = row['file']
        df_image['img_path'] = img_path
        df_image['mask_path'] = mask_path
        intensity_scaled_results.append(df_image)
        
    except Exception as e:
        print(f"Error processing file {row['file']}: {e}")
        continue

# Combine all results
if intensity_scaled_results:
    intensity_scaled_data = pd.concat(intensity_scaled_results, ignore_index=True)
    print(f"Successfully processed {len(intensity_scaled_data)} regions")
    #print(intensity_scaled_data.head())
else:
    print("No data processed")

## Reannotate data

In [ ]:
# Replace data with intensity scaled data completely
data = intensity_scaled_data

In [ ]:
data['experiment'] = data['file'].apply(lambda x: '_'.join(x.split('_')[:2]))
data['experiment'] = data['experiment'].replace({'231210_LC3': '231210_LC1'})
data['mouse_id'] = data['experiment'].apply(lambda x: x.split('_')[1])
data['mouse_id'] = data['mouse_id'].replace({'LC1': 'WT1', 'LC2': 'WT2', 'LC3': 'WT3', 'LC6': 'WT6',
                                           'DB2': 'DB2', 'DB6': 'DB6', 'DB1':'DB1'})
data['condition'] = data['experiment'].apply(lambda x: x.split('_')[1][:-1])
data['condition'] = data['condition'].replace({'LC': 'WT'})
data['treatment'] = data['file'].apply(lambda x: x.split('_')[2])
data['tech_replicate'] = data['file'].apply(lambda x: x.split('_')[3])


In [ ]:
data['biological_replicate'] = data['mouse_id'].map({'WT1': 1, 'WT2': 2, 'WT3': 3, 'WT6': 4,
                                                     'DB2': 1, 'DB6': 2, 'DB1': 3})

In [ ]:
# Calculate max width and height for gallery display from the data bboxes
data['height'] = data['maxr'] - data['minr']
data['width'] = data['maxc'] - data['minc']
MAX_H = int(data['height'].max())
MAX_W = int(data['width'].max())
print(f"Max height: {MAX_H}, Max width: {MAX_W}")

## Compute projections on metabolic trajectory

### Calculate tau bound with linear regression through all phasor points

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

def get_free_bound_nadh_coords(data, tau_free_s=0.4*1e-9):
    g_free_est = 1 / (1 + (tau_free_s * OMEGA)**2)
    s_free_est = (tau_free_s * OMEGA) / (1 + (tau_free_s * OMEGA)**2)

    g = data["g_corrected"].values
    s = data["s_corrected"].values
    w = data["total_photon_counts"].values

    # SHIFT so the estimated free NADH point is at the origin
    Xs = (g - g_free_est).reshape(-1, 1)
    ys = (s - s_free_est)

    # weighted regression through origin in shifted space
    model = LinearRegression(fit_intercept=False)
    model.fit(Xs, ys, sample_weight=w)

    slope = model.coef_[0]
    intercept = model.intercept_
    print(f"Weighted LinearRegression: slope={slope}, intercept={intercept}")
    # back-transform to original coordinates:
    # s - s_free = slope (g - g_free)  ->  s = slope*g + (s_free - slope*g_free)
    intercept = s_free_est - slope * g_free_est

    print(f"Weighted LinearRegression: slope={slope}, intercept={intercept}")

    # intersection with universal semicircle: (g-0.5)^2 + s^2 = 0.25
    a = 1 + slope**2
    b = 2*slope*intercept - 1
    c = intercept**2
    disc = b*b - 4*a*c
    if disc < 0:
        raise ValueError("No intersection between regression line and universal semicircle.")

    sqrt_disc = np.sqrt(disc)
    g_roots = [(-b + sqrt_disc)/(2*a), (-b - sqrt_disc)/(2*a)]
    pts = []
    for gr in g_roots:
        sr = slope*gr + intercept
        if 0 <= gr <= 1 and sr >= 0:   # upper semicircle + valid range
            pts.append((gr, sr))

    if len(pts) == 0:
        raise ValueError("Intersections exist algebraically but not on the upper semicircle in [0,1].")

    # free NADH = shorter lifetime = higher g (closer to 1)
    pts = sorted(pts, key=lambda t: t[0], reverse=True)
    free = pts[0]
    bound = pts[-1] if len(pts) > 1 else None

    return free, bound, (slope, intercept)


In [ ]:
data_fit = data[data['treatment'].isin(['NA', 'AA', 'CA'])]
free_nadh_coords, bound_nadh_coords, (slope, intercept) = get_free_bound_nadh_coords(data_fit)
tau_bound = 1/OMEGA* (bound_nadh_coords[1]/bound_nadh_coords[0]) # in s
# Calculate g and s for theoretical bound NADH lifetimes (3.4 ns)
g_bound_theoretical = 1 / (1 + (3.4e-9 * OMEGA)**2)
s_bound_theoretical = (3.4e-9 * OMEGA) / (1 + (3.4e-9 * OMEGA)**2)
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
print(f"Free NADH coords: G={free_nadh_coords[0]:.4f}, S={free_nadh_coords[1]:.4f}")
print(f"Bound NADH coords: G={bound_nadh_coords[0]:.4f}, S={bound_nadh_coords[1]:.4f}")
print(f"Phase lifetime of free NADH: {1/OMEGA* (free_nadh_coords[1]/free_nadh_coords[0])*1e9:.2f} ns")
print(f"Phase lifetime of bound NADH: {1/OMEGA* (bound_nadh_coords[1]/bound_nadh_coords[0])*1e9:.2f} ns")
# Plot phasor plot with data points and free/bound NADH coords
plt.figure(figsize=(3.5,1.8))
# Add universal semicircle
theta = np.linspace(0, np.pi/2, 100)
u = np.cos(theta)**2
v = np.sin(theta)*np.cos(theta)
plt.plot(u, v, color='black', linestyle='--', label='Universal Semicircle')
plt.xlim(0, 1)
plt.ylim(0, 0.5)
plt.xlabel('G')
plt.ylabel('S')
#plt.title('Phasor Plot with Data Points and Free/Bound NADH')

# Plot data points
plt.scatter(data_fit['g_corrected'], data_fit['s_corrected'], s=10, color=palette[0], alpha=0.5, label='Data Points')

# Plot the regression line
x_vals = np.linspace(bound_nadh_coords[0], free_nadh_coords[0], 100)
y_vals = slope * x_vals + intercept
plt.plot(x_vals, y_vals, color=palette[3], linestyle='--', linewidth=2, label='Regression Line')


# Plot free and bound NADH points
plt.scatter(free_nadh_coords[0], free_nadh_coords[1], color=palette[1], s=100, label=f'Free NAD(P)H\nτ={TAU_FREE*1e9:.2f} ns', edgecolor='black')
plt.scatter(g_bound_theoretical, s_bound_theoretical, color='gray', s=50, label='LDH Bound NAD(P)H\nτ=3.4 ns', edgecolor='black', marker='o')

plt.scatter(bound_nadh_coords[0], bound_nadh_coords[1], color=palette[2], s=100, label=f'Bound NAD(P)H\nτ={tau_bound*1e9:.2f} ns', edgecolor='black')
plt.legend(loc='center left', fontsize=10, bbox_to_anchor=(1.05,0.5))


# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Phasor_plot_with_free_bound_NADH.png'), bbox_inches='tight', dpi=300)
plt.show()


### Compute Metabolic index

In [ ]:
tau_free = TAU_FREE
tau_bound = tau_bound if FIT_TAU_BOUND else TAU_BOUND
omega = OMEGA  # rad/s
g_free = 1 / (1 + (omega * tau_free)**2)
g_bound = 1 / (1 + (omega * tau_bound)**2)
s_free = (omega * tau_free) / (1 + (omega * tau_free)**2)
s_bound = (omega * tau_bound) / (1 + (omega * tau_bound)**2)

P_free = np.array([g_free, s_free])
P_bound = np.array([g_bound, s_bound])
metabolic_trajectory_vector = P_bound - P_free

In [ ]:
# Compute bound fraction
data['fraction_bound'] = (
    ((data['g_corrected'] - g_free) * metabolic_trajectory_vector[0] +
     (data['s_corrected'] - s_free) * metabolic_trajectory_vector[1]) /
    (metabolic_trajectory_vector @ metabolic_trajectory_vector)
)

In [ ]:
# Compute distance to metabolic trajectory
def compute_distance_to_trajectory(row, P_free, metabolic_trajectory_vector):
    P_cell = np.array([row['g_corrected'], row['s_corrected']])
    P_free_to_cell = P_cell - P_free
    projection_length = np.dot(P_free_to_cell, metabolic_trajectory_vector) / np.linalg.norm(metabolic_trajectory_vector)
    projection_vector = (projection_length / np.linalg.norm(metabolic_trajectory_vector)) * metabolic_trajectory_vector
    orthogonal_vector = P_free_to_cell - projection_vector
    distance = np.linalg.norm(orthogonal_vector)
    return distance
data['distance_to_trajectory'] = data.apply(
    lambda row: compute_distance_to_trajectory(row, P_free, metabolic_trajectory_vector), axis=1)

In [ ]:
data['distance_to_trajectory'].describe()

## Choose feature thresholds for data inclusion

In [ ]:
area_threshold = 100  # um²
median_photon_counts_threshold = 25 # photons
total_photon_counts_threshold = 500  # photons
metabolic_deviation_threshold = 0.05  # arbitrary units  
eccentricity_threshold = 0.2  # dimensionless
area_eccentricity_threshold = (200, 0.25)  # min area, min eccentricity

In [ ]:
# Rename columns for better display
data_renamed = data.rename(columns={'treatment': 'Treatment',
                                    'area': 'Area (μm²)',
                                    'median_photon_counts': 'Median Photon Counts',
                                    'fraction_bound': 'Metabolic Index',
                                    'distance_to_trajectory': 'Metabolic Deviation',
                                    'eccentricity': 'Eccentricity',
                                    'solidity': 'Solidity',
                                    'total_photon_counts': 'Total Photon Counts',
                                    'tech_replicate': 'Technical Replicate',
                                    'biological_replicate': 'Biological Replicate'})
# Configure set of thresholds to plot dashed lines for each feature
thresholds = {
    'Area (μm²)': area_threshold,
    'Median Photon Counts': median_photon_counts_threshold,
    'Metabolic Index': None,
    'Metabolic Deviation': metabolic_deviation_threshold,
    'Eccentricity': eccentricity_threshold,
    'Solidity': None,
    'Total Photon Counts': total_photon_counts_threshold,
}
arrows = {
    'Area (μm²)': 'right',
    'Median Photon Counts': 'right',
    'Metabolic Index': None,
    'Metabolic Deviation': 'left',
    'Eccentricity': 'right',
    'Solidity': None,
    'Total Photon Counts': 'right',
}

# Sort by treatment for consistent colors
treatment_order = {'NA': 1, 'AA': 2, 'CA': 3, 'ES1': 4, 'ES2': 5, 'ES3': 6, 'ES4': 7}
data_renamed['Treatment_order'] = data_renamed['Treatment'].map(treatment_order)
data_renamed = data_renamed.sort_values('Treatment_order')

In [ ]:
# Annotate included/excluded regions based on area and eccentricity thresholds
data_renamed['excluded'] = ((data_renamed['Area (μm²)'] < area_threshold) | 
                            (data_renamed['Eccentricity'] < eccentricity_threshold) |
                            ((data_renamed['Eccentricity'] < area_eccentricity_threshold[1]) & (data_renamed['Area (μm²)'] < area_eccentricity_threshold[0])) |
                            (data_renamed['Median Photon Counts'] < median_photon_counts_threshold) |
                            (data_renamed['Total Photon Counts'] < total_photon_counts_threshold) |
                            (data_renamed['Metabolic Deviation'] > metabolic_deviation_threshold)#
                            #|((data_renamed['Median Photon Counts'] < area_mean_photon_counts_threshold[1]) & (data_renamed['Area (μm²)'] < area_mean_photon_counts_threshold[0]))
                            )
data_renamed['inclusion'] = data_renamed['excluded'].apply(lambda x: 'Excluded' if x else 'Included')

In [ ]:
# Create excluded reason column
def get_exclusion_reasons(row):
    reasons = []
    if row['excluded'] == False:
        return 'Included'
    if row['Area (μm²)'] < area_threshold:
        reasons.append('Area < threshold')
    if row['Eccentricity'] < eccentricity_threshold:
        reasons.append('Eccentricity < threshold')
    if (row['Eccentricity'] < area_eccentricity_threshold[1]) & (row['Area (μm²)'] < area_eccentricity_threshold[0]):
        reasons.append('Area & Eccentricity combo < threshold')
    if row['Median Photon Counts'] < median_photon_counts_threshold:
        reasons.append('Median Photon Counts < threshold')
    if row['Total Photon Counts'] < total_photon_counts_threshold:
        reasons.append('Total Photon Counts < threshold')
    if row['Metabolic Deviation'] > metabolic_deviation_threshold:
        reasons.append('Metabolic Deviation > threshold')
    return '; '.join(reasons) if reasons else 'Included'
data_renamed['exclusion_reasons'] = data_renamed.apply(get_exclusion_reasons, axis=1)


In [ ]:
# Get % of excluded cells per treatment and technical replicate and save report as text file
print("Exclusion report per treatment and technical replicate:")
with open(os.path.join(SAVE_DIR, f'exclusion_report_{today.strftime("%d%m%Y")}.txt'), 'w') as report_file:
    for (condition, mouse_id, treatment, experiment, tech_rep), group in data_renamed.groupby(['condition','mouse_id','Treatment', 'experiment','Technical Replicate']):
        n_total = len(group)
        n_excluded = group['excluded'].sum()
        excluded_pct = (n_excluded / n_total) * 100
        report_file.write(f"Condition: {condition}, Mouse ID: {mouse_id}, Treatment: {treatment}, Experiment: {experiment}, Technical Replicate: {tech_rep} - Excluded {n_excluded} out of {n_total} cells ({excluded_pct:.2f}%)\n")
        print(f"Condition: {condition}, Mouse ID: {mouse_id}, Treatment: {treatment}, Experiment: {experiment}, Technical Replicate: {tech_rep} - Excluded {n_excluded} out of {n_total} cells ({excluded_pct:.2f}%)")
        # Add exclusion reason %
        exclusion_reasons_counts = group[group['excluded']==True]['exclusion_reasons'].value_counts()
        for reason, count in exclusion_reasons_counts.items():
            reason_pct = (count / n_excluded) * 100
            report_file.write(f"    {reason}: {count} cells ({reason_pct:.2f}%)\n")
            print(f"    {reason}: {count} cells ({reason_pct:.2f}%)")
    excluded_percentage = (data_renamed['excluded'].sum() / len(data_renamed)) * 100
    report_file.write(f"\nOverall Exclusion: Excluded {data_renamed['excluded'].sum()} out of {len(data_renamed)} cells ({excluded_percentage:.2f}%) based on quality criteria.\n")
    print(f"\nOverall Exclusion: Excluded {data_renamed['excluded'].sum()} out of {len(data_renamed)} cells ({excluded_percentage:.2f}%) based on quality criteria.")   


# Save final data with inclusion/exclusion annotation
data_renamed.to_csv(os.path.join(SAVE_DIR, f'FLIM_phasor_data_corrected_annotated_{today.strftime("%d%m%Y")}.csv'), index=False)

In [ ]:
# Plot boxen plots of area, fraction bound, photon counts, distance to trajectory, and total photon counts in the same figure with seaborn
sns.set_context('paper', font_scale=1)
sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 3, figsize=(5, 2.5))
#plt.subplots_adjust(hspace=1, wspace=0.05)  # Make space on the right for global legend
# Store first axis for legend extraction
first_ax = None
features = ['Area (μm²)', 'Eccentricity', 'Metabolic Deviation', 'Median Photon Counts', 'Total Photon Counts' , 'Skip']
for ax, feature in zip(axes.flatten(), features):
    if feature == 'Skip':
        ax.axis('off')
        continue
    g = sns.boxenplot(data_renamed, x=feature, hue='Treatment', palette='Set2', ax=ax, legend=True if first_ax is None else False)
    if first_ax is None:
        first_ax = ax
    #g.set_title(f'Distribution of\n{feature.replace("_", " ").title()}')
    
    g.set_title('')
    g.set_xlabel(feature.replace("_", " "))
    g.set_ylabel('')
    # if feature == 'Median Photon Counts':
    #     g.set_xlim(0, 500)
    # Add dashed lines for thresholds
    if feature in thresholds:
        
        if feature == 'Total Photon Counts':
            g.set_xscale('log')
            g.set_xlabel('Total Photon Counts\n(log scale)', labelpad=8)
        thr = thresholds[feature]
        if thr is not None:
            ax.axvline(thr, color='red', linestyle='--', linewidth=1)
            # Add arrow annotation
            if arrows[feature] == 'left':
                ax.annotate(#'Upper\nthreshold', 
                            '',
                            ha ='left',
                            va ='center',
                            color='red',
                            fontsize=8,
                            xy=(thr-ax.get_xlim()[1]*0.2, ax.get_ylim()[1]*0.85), 
                            xytext=(thr*1.02, ax.get_ylim()[1]*0.85),
                            arrowprops=dict(color='red', arrowstyle='<-'))
            elif arrows[feature] == 'right':
                ax.annotate(#'Lower\nthreshold', 
                            '',
                            ha ='right',
                            va ='center',
                            color='red',
                            fontsize=8,
                            xy=(thr+ax.get_xlim()[1]*0.2, ax.get_ylim()[1]*0.85), 
                            xytext=(thr*0.9, ax.get_ylim()[1]*0.85), 
                            arrowprops=dict(color='red', arrowstyle='<-'))
        if feature == 'Median Photon Counts':
            g.set_xscale('log')
            g.set_xlabel('Median Photon Counts\n(log scale)', labelpad=8)

# Extract legend from first plot and create global legend
handles, labels = first_ax.get_legend_handles_labels()
first_ax.get_legend().remove()  # Remove the legend from first subplot
#fig.legend(handles, labels, title='Treatment', bbox_to_anchor=(0.9, 0.5), loc='upper center')

plt.tight_layout()
# Save and show plot
fig.savefig(os.path.join(SAVE_DIR, f'Fig1-b_boxen_cell_properties_{today.strftime("%d%m%Y")}.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.ticker import AutoMinorLocator
# Plot pairplot of all thresholded features, hue by excluded/included
sns.set_context('paper', font_scale=1.2)
sns.set_style('whitegrid')

g=sns.PairGrid(data_renamed, height=1.5, aspect=1.2,
            vars=['Area (μm²)', 'Eccentricity', 'Median Photon Counts','Total Photon Counts', 'Metabolic Deviation'],
            hue='inclusion', hue_order=['Included', 'Excluded'],
            palette = {'Excluded': "#f77f39", 'Included': "#599cff"}, 
            corner=True).map_diag(sns.histplot, kde=True)
g.map_lower(sns.scatterplot, s=1, alpha=0.5).add_legend(title='Analysis Inclusion', markerscale=10, bbox_to_anchor=(0.6, 0.85), loc='center left')
labels = ['Area (μm²)', 'Eccentricity\n', 'Median\nPhoton Counts', 'Total\nPhoton Counts', 'Metabolic\nDeviation']
for i, ax in enumerate(g.axes.flatten()):
    if ax is not None:
        ax.set_xlabel(labels[i % 5])
        ax.set_ylabel(labels[i // 5])
        # Add minor grid lines and ticks
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.grid(which='both', axis='x', linestyle='--', linewidth=0.5)
        ax.grid(which='both', axis='y', linestyle='--', linewidth=0.5)
        ax.tick_params(which='major', axis='x', direction='out', bottom=True, length=2)
        ax.tick_params(which='major', axis='y', direction='out', left=True, length=2)
    # Make x-scale log for total photon counts
    if (i % 5 == 3) and (i // 5 != i % 5):
        if ax not in [None]:
            ax.set_xscale('log')
            ax.set_xlabel('Total Photon\nCounts (log scale)', labelpad=1, fontsize=12)
    # Set y-scale log for total photon counts
    if (i // 5 == 3) and (i // 5 != i % 5):
        if ax not in [None]:
            ax.set_yscale('log')
            ax.set_ylabel('Total Photon\nCounts (log scale)', labelpad=1, fontsize=12)
    # Make x-scale log for median photon counts
    if (i % 5 == 2) and (i // 5 != i % 5):
        if ax not in [None]:
            ax.set_xscale('log')
            ax.set_xlabel('Median Photon\nCounts (log scale)', labelpad=1, fontsize=12)
    # Set y-scale log for median photon counts
    if (i // 5 == 2) and (i // 5 != i % 5):
        if ax not in [None]:
            ax.set_yscale('log')
            ax.set_ylabel('Median Photon\nCounts (log scale)', labelpad=1, fontsize=12)
        
#plt.suptitle('Pairplot of Cell Properties and Metabolic Parameters', y=1.02)


plt.savefig(os.path.join(SAVE_DIR, f'Fig1-c_PairplotFeatures_{today.strftime("%d%m%Y")}.png'), bbox_inches='tight', dpi=300)


## Plot representative galleries of cells excluded/included in the analysis

In [ ]:
for feature, threshold in thresholds.items():
    print(f"{feature} threshold: {threshold}")
    if feature == 'Metabolic Deviation':
        to_remove = data_renamed[data_renamed[feature] > threshold]
    else:
        to_remove = data_renamed[data_renamed[feature] < threshold]
    print(f"Number of cells excluded based on {feature}: {len(to_remove)}")
    try:
        plot_representative_cells(to_remove, savedir=SAVE_DIR, n_cells=5, 
                                lifetime=True, photons=True, auto_contrast=True, 
                                label=f'excluded_{feature.replace(" ", "_").lower()}', add_colorbar=True, 
                                feature=feature, area_feature='Area (μm²)',
                                ncols=5)
    except Exception as e:
        print(f"Error plotting representative cells for {feature}: {e}")

In [ ]:
to_remove = data_renamed[(data_renamed['Area (μm²)']<area_eccentricity_threshold[0]) & (data_renamed['Eccentricity']<area_eccentricity_threshold[1])]
print(f"Number of cells excluded based on area & eccentricity combo: {len(to_remove)}")
plot_representative_cells(to_remove, savedir=SAVE_DIR, n_cells=5,
                            lifetime=True, photons=True, auto_contrast=True, 
                            label=f'excluded_area_eccentricity_combo', add_colorbar=True, 
                            feature='Eccentricity', area_feature='Area (μm²)',
                            ncols=5)

In [ ]:
# Plot representative cells from each condition&treatment group
for g in data_renamed[data_renamed['excluded']==False].groupby(['condition', 'Treatment']):
    group_name = f"{g[0][0]}_{g[0][1]}"
    group_df = g[1]
    print(f"Plotting representative cells for group: {group_name} with {len(group_df)} cells.")
    plot_representative_cells(group_df, SAVE_DIR,
                              n_cells=4, ncols=4,
                              lifetime=True, photons=True, auto_contrast=True, 
                              label=group_name, feature='Metabolic Index', area_feature='Area (μm²)')
    

In [ ]:
data_renamed.rename(columns={'biological_replicate': 'Biological Replicate'}, inplace=True)
data_to_analyse = data_renamed[data_renamed['excluded']==False]


## Plot phasor diagrams

In [ ]:
condition_sort = {'WT': 1, 'DB': 2}
treatment_sort = {'NA': 1, 'AA': 2, 'CA': 3, 'ES1': 4, 'ES2': 5, 'ES3': 6, 'ES4': 7}
data_to_analyse['condition_order'] = data_to_analyse['condition'].map(condition_sort)
data_to_analyse['treatment_order'] = data_to_analyse['Treatment'].map(treatment_sort)
data_to_analyse.sort_values(['condition_order', 'treatment_order', 'Biological Replicate'], inplace=True)
summary_stats = data_to_analyse[['condition','Biological Replicate','Treatment','tau','Metabolic Index','Metabolic Deviation','Median Photon Counts']].groupby(['condition','Biological Replicate','Treatment']).describe()
# Sort summary stats by condition and treatment
summary_stats = summary_stats.reset_index()
summary_stats['condition_order'] = summary_stats['condition'].map(condition_sort)
summary_stats['treatment_order'] = summary_stats['Treatment'].map(treatment_sort)
summary_stats.sort_values(['condition_order', 'treatment_order', 'Biological Replicate'], inplace=True)
summary_stats.drop(columns=['condition_order', 'treatment_order'], inplace=True)
summary_stats.to_csv(os.path.join(SAVE_DIR, 'summary_statistics_per_condition_treatment.csv'))


In [ ]:
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

g = sns.catplot(
    data=data_to_analyse,
    x='Treatment',
    col='condition',
    col_order=['WT', 'DB'],
    hue='Biological Replicate',
    palette='Set2',
    kind='count',
    height=3,
    aspect=1.2
)
g.set_titles(col_template='{col_name}')

g.set_axis_labels('', 'Cell Count')

plt.savefig(
    os.path.join(SAVE_DIR, 'Fig2_Cell_count_per_Treatment_Biological_Replicate.png'),
    dpi=300, bbox_inches='tight'
)


In [ ]:
# Create stacked bar countplot using seaborn histplot
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1)

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(5, 5), sharex=False)
for condition, ax in zip(['WT','DB'], axs):
    sns.histplot(data=data_to_analyse[data_to_analyse['condition']==condition], x='Treatment', 
                 hue='Biological Replicate', legend=False, stat='count',
                multiple='stack', discrete=True, palette='Set2', ax=ax)

    ax.set_xlabel('') 
    ax.set_ylabel('Cell Count', fontweight='bold')
    ax.set_title(f'{condition}', fontweight='bold', fontsize=16, pad=5)
# Get unique biological replicates and create custom legend
bio_reps = sorted(data_to_analyse['Biological Replicate'].unique())
colors = sns.color_palette('Set2', n_colors=len(bio_reps))

# Create legend handles manually
handles = [plt.Rectangle((0,0),1,1, color=colors[i]) for i in range(len(bio_reps))]
labels = [f'{rep}' for rep in bio_reps]

ax.legend(handles, labels, title='Biological\nReplicate', 
        bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'Fig2-b_Cell_counts_per_condition_treatment.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Plot ES1 mean bound fraction per file
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)
plt.figure(figsize=(8,4))
sns.barplot(data=data[data['treatment']=='ES1'], 
            x='file', y='fraction_bound', 
            hue='condition',
            palette='Set2',
            ci='sd')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Mean Lifetime (ns)')
plt.title('Mean bound fraction per File for ES1 Treatment')
plt.legend(title='Condition')


In [ ]:
# Plot Facet Grid of phasor plots per condition and treatment
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.5)
custom_treatment_palette = {
    'NA': "#a522e2",  # golden yellow
    'AA': "#5aff0e",  # green fluorescent
    'CA': "#e96c19",  # bright orange
    'ES1': "#27d6b0",  # turquoise
    'ES2': "#1aceee",  # vivid sky blue
    'ES3': "#0753f7",  # royal blue
    'ES4': "#9802fc"   # vibrant purple
}
controls = data_to_analyse[data_to_analyse['Treatment'].isin(['NA'])]
es_treated = data_to_analyse[data_to_analyse['Treatment'].isin(['ES1', 'ES2', 'ES3', 'ES4'])]
g = sns.FacetGrid(controls, row="condition", 
                  #col="Treatment", 
                  hue='Treatment',
                  margin_titles=False, 
                  row_order=['WT', 'DB'], col_order=[1, 2, 3, 4], palette='Set2',
                  height=2, aspect=2, despine=True)
# Add universal semicircle and metabolic trajectory line
theta = np.linspace(0, np.pi/2, 100)
u = np.cos(theta)**2
v = np.sin(theta)*np.cos(theta)
def plot_semicircle(*args, **kwargs):
    plt.plot(u, v, color='black', linestyle='--', label='Universal Semicircle')
    # Plot metabolic trajectory line
    plt.plot([g_free, g_bound], [s_free, s_bound], color='red', linestyle=':', label='Metabolic Trajectory')
    plt.scatter([g_free, g_bound], [s_free, s_bound], color='blue')
    plt.text(g_free, s_free, f'Free NADH\nτ={tau_free*1e9:.2f} ns', fontsize=10, verticalalignment='bottom', horizontalalignment='left')
    plt.text(g_bound, s_bound, f'Bound NADH\nτ={tau_bound*1e9:.2f} ns', fontsize=10, verticalalignment='bottom', horizontalalignment='right')
g.map(plot_semicircle)
g.map_dataframe(sns.scatterplot, x='g_corrected', y='s_corrected', alpha=0.8, 
                s=10, 
                #levels=10,
                #hue='Treatment', palette='Set1', 
                legend='full')
g.set_axis_labels('G ', 'S ')
g.set_titles(row_template='{row_name}', col_template='Biological Replicate {col_name}')
g.add_legend(markerscale=5, title='Treatment', bbox_to_anchor=(0.9, 0.6), loc='center left')
plt.subplots_adjust(top=0.9)
# Remove axes with no data
for ax in g.axes.flatten():
    if not ax.has_data():
        ax.remove()
#g.fig.suptitle('Phasor Plots by Condition and Treatment')
plt.savefig(os.path.join(SAVE_DIR, 'Fig3_Control_phasor_plots.png'), bbox_inches='tight', dpi=300)


In [ ]:
# Plot Facet Grid of phasor plots per condition and treatment
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.5)

custom_treatment_palette = sns.color_palette("Set2", n_colors=7)
custom_treatment_palette = custom_treatment_palette[-4:]
g = sns.FacetGrid(es_treated, row="condition", 
                  #col="biological_replicate", 
                  hue='Treatment',
                  margin_titles=False, 
                  row_order=['WT', 'DB'], col_order=[1, 2, 3, 4], palette=custom_treatment_palette,
                  height=2, aspect=2, despine=True)
# Add universal semicircle and metabolic trajectory line
theta = np.linspace(0, np.pi/2, 100)
u = np.cos(theta)**2
v = np.sin(theta)*np.cos(theta)
def plot_semicircle(*args, **kwargs):
    plt.plot(u, v, color='black', linestyle='--', label='Universal Semicircle')
    # Plot metabolic trajectory line
    plt.plot([g_free, g_bound], [s_free, s_bound], color='red', linestyle=':', label='Metabolic Trajectory')
    plt.scatter([g_free, g_bound], [s_free, s_bound], color='blue')
    plt.text(g_free, s_free, f'Free NADH\nτ={tau_free*1e9:.2f} ns', fontsize=10, verticalalignment='bottom', horizontalalignment='left')
    plt.text(g_bound, s_bound, f'Bound NADH\nτ={tau_bound*1e9:.2f} ns', fontsize=10, verticalalignment='bottom', horizontalalignment='right')
g.map(plot_semicircle)
g.map_dataframe(sns.scatterplot, x='g_corrected', y='s_corrected', alpha=0.9, s=10, 
                #hue='treatment', palette='Set1', 
                legend='full')
g.set_axis_labels('G ', 'S ')
g.set_titles(row_template='{row_name}', col_template='Biological Replicate {col_name}')
g.add_legend(markerscale=5, title='Treatment', bbox_to_anchor=(0.9, 0.6), loc='center left')
plt.subplots_adjust(top=0.9)
# Remove axes with no data
for ax in g.axes.flatten():
    if not ax.has_data():
        ax.remove()
#g.fig.suptitle('Phasor Plots by Condition and Treatment')
plt.savefig(os.path.join(SAVE_DIR, 'Fig3-b_ES_treated_phasor_plots.png'), bbox_inches='tight', dpi=300)


In [ ]:
# Plot Facet Grid of phasor plots per condition and treatment
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.5)
palette = sns.color_palette("Set2", n_colors=7)

for i, tr in enumerate(['NA','AA','CA','ES1', 'ES2', 'ES3', 'ES4']):
    custom_treatment_palette = [palette[0],palette[i]]
    if tr == 'NA':
        comparison = data_to_analyse[data_to_analyse['Treatment'].isin(['NA'])]
        custom_treatment_palette = [palette[0]]
    else:
        comparison = data_to_analyse[data_to_analyse['Treatment'].isin(['NA', tr])]
    g = sns.FacetGrid(comparison, row="condition", 
                    #col="biological_replicate", 
                    hue='Treatment',
                    margin_titles=False, 
                    row_order=['WT', 'DB'], 
                    #col_order=[1, 2, 3, 4], 
                    palette=custom_treatment_palette,
                    height=2, aspect=2, despine=True
                    )
    # Add universal semicircle and metabolic trajectory line
    theta = np.linspace(0, np.pi/2, 100)
    u = np.cos(theta)**2
    v = np.sin(theta)*np.cos(theta)
    def plot_semicircle(*args, **kwargs):
        plt.plot(u, v, color='black', linestyle='--', label='Universal Semicircle')
        # Plot metabolic trajectory line
        plt.plot([g_free, g_bound], [s_free, s_bound], color='red', linestyle=':', label='Metabolic Trajectory')
        plt.scatter([g_free, g_bound], [s_free, s_bound], color='red')
        plt.annotate(
            f'Free NAD(P)H\nτ={tau_free*1e9:.2f} ns',
            xy=(g_free, s_free),
            xytext=(1.2*g_free, 1.2*s_free),
            textcoords='offset points',
            fontsize=10,
            ha='left',
            va='bottom',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.1)
        )

        plt.annotate(
            f'Bound NAD(P)H\nτ={tau_bound*1e9:.2f} ns',
            xy=(g_bound, s_bound),
            xytext=(0.8*g_bound, 1.2*s_bound),
            textcoords='offset points',
            fontsize=10,
            ha='right',
            va='bottom',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.1)
        )

    g.map(plot_semicircle)
    g.map_dataframe(sns.scatterplot, x='g_corrected', y='s_corrected', alpha=0.9, s=3, 
                    #hue='treatment', palette='Set1', 
                    legend='full')
    g.set_axis_labels('G ', 'S ')
    g.set_titles(row_template='{row_name}', col_template='Biological Replicate {col_name}')
    g.add_legend(markerscale=5, title='Treatment', bbox_to_anchor=(0.8, 0.6), loc='center left')
    plt.subplots_adjust(top=0.9)
    # Remove axes with no data
    for ax in g.axes.flatten():
        if not ax.has_data():
            ax.remove()
    #g.fig.suptitle('Phasor Plots by Condition and Treatment')
    plt.savefig(os.path.join(SAVE_DIR, f'Fig4_Phasors_{tr}.png'), bbox_inches='tight', dpi=300)


## Metabolic index stats

In [ ]:
# Aggregate by condition, biological replicate, and treatment to get median Metabolic Index and counts
stats_data = data_to_analyse.groupby(['condition', 'Biological Replicate', 'Treatment']).agg(
    median_metabolic_index = ('Metabolic Index', 'median'),
    cell_count = ('Metabolic Index', 'count')).reset_index()
# Save stats data
stats_data.to_csv(os.path.join(SAVE_DIR, f'Fig5_Stats_data.csv'), index=False)


In [ ]:
# Apply Friedman test to compare Metabolic Index across treatments within each condition
from scipy.stats import friedmanchisquare
treatments_to_compare = ['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
results = []
for condition in ['WT', 'DB']:
    condition_data = stats_data[(stats_data['condition'] == condition) &
                                 (stats_data['Treatment'].isin(treatments_to_compare))]
    # Pivot data to have treatments as columns
    stats_data_wide = condition_data.pivot_table(index=['condition', 'Biological Replicate'],
                                                columns='Treatment', values='median_metabolic_index')
    # Apply Friedman test
    stat, p = friedmanchisquare(*[stats_data_wide[col] for col in stats_data_wide.columns])
    # Store results
    results_df = pd.DataFrame({
        'condition': [condition],
        'friedman_statistic': [stat],
        'p_value': [p]
    })
    results.append(results_df)

    print(f"Friedman test for {condition}: stat={stat}, p={p}")
# Combine results
results_df = pd.concat(results, ignore_index=True)
results_df.to_csv(os.path.join(SAVE_DIR, f'Fig5_Friedman_test_results.csv'), index=False)

# Apply paired post-hoc Wilcoxon tests with Holm correction between treatments within each condition
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import itertools
results = []
for condition in ['WT', 'DB']:
    condition_data = stats_data[(stats_data['condition'] == condition) &
                                 (stats_data['Treatment'].isin(treatments_to_compare))]
    # Pivot data to have treatments as columns
    stats_data_wide = condition_data.pivot_table(index=['condition', 'Biological Replicate'],
                                                columns='Treatment', values='median_metabolic_index')
    # Perform pairwise Wilcoxon tests
    #treatment_pairs = list(itertools.combinations(treatments_to_compare, 2))
    # Only compare each AA/CA/ES treatment to NA control, plus AA vs CA
    treatment_pairs = [(treatments_to_compare[0], tr) for tr in treatments_to_compare[1:]] + [('AA', 'CA')]
    print(f"Post-hoc Wilcoxon tests for {condition}:")
    for pair in treatment_pairs:
        try:
            stat, p = wilcoxon(stats_data_wide[pair[0]], stats_data_wide[pair[1]])
            # Apply Holm correction later, but store uncorrected p-values for now
            # store pair, stat and p in df
            df_result = pd.DataFrame({'Condition': [condition],
                                      'Treatment 1': [pair[0]],
                                      'Treatment 2': [pair[1]],
                                      'Wilcoxon stat': [stat],
                                      'p-value': [p]})
            results.append(df_result)

            print(f"  {pair[0]} vs {pair[1]}: stat={stat}, p={p}")
        except ValueError as e:
            print(f"  {pair[0]} vs {pair[1]}: Could not perform test ({e})")
    # Apply Holm correction to p-values for this condition
    p_values = [r['p-value'].values[0] for r in results if r['Condition'].values[0] == condition]
    reject, pvals_corrected, _, _ = multipletests(p_values, method='holm')
    # Update results with corrected p-values
    idx = 0
    for i in range(len(results)):
        if results[i]['Condition'].values[0] == condition:
            results[i]['p-value (Holm corrected)'] = pvals_corrected[idx]
            results[i]['Reject Null'] = reject[idx]
            idx += 1
            
# Combine results into a single DataFrame
results_df = pd.concat(results, ignore_index=True)
results_df.to_csv(os.path.join(SAVE_DIR, f'Fig5_Wilcoxon_posthoc_results.csv'), index=False)

In [ ]:
g=sns.FacetGrid(data_to_analyse, row='condition', row_order=['WT', 'DB'],
                height=2.8, aspect=2, sharex=True, despine=True)
g.map_dataframe(
    sns.violinplot, y='Metabolic Index', x='Treatment', hue='Biological Replicate', 
    palette='Set2', cut=0, dodge=True,
    inner='quartiles', alpha=1, color = 'gray', order=['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
).add_legend(title='Biological\nReplicate', bbox_to_anchor=(0.8, 0.5), loc='center left')
# g.map_dataframe(
#     sns.swarmplot, y='Metabolic Index', x='Treatment', 
#     hue='Biological Replicate', size=0.7, alpha=0.9, palette = 'Set2'
# ).add_legend(title='Biological Replicate', markerscale=10, bbox_to_anchor=(0.85, 0.5), loc='center left')
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_xlabels('')
g.set_ylabels('Metabolic Index')
#Statistical annotation

pairs = [
    ('NA', 'AA'),
    ('NA', 'CA'),
    ('AA', 'CA'),
    ('NA', 'ES1'),
    ('NA', 'ES2'),
    ('NA', 'ES3'),
    ('NA', 'ES4'),
]

all_results = []
for ax, condition in zip(g.axes.flat, g.row_names):
    xticks = ax.get_xticks()
    for i, x in enumerate(xticks):
        if i % 2 == 0:
            ax.axvspan(
                x - 0.5,
                x + 0.5,
                color="0.95",
                zorder=0
            )
    ax.set_axisbelow(True)
    data_subset = data_to_analyse[data_to_analyse['condition'] == condition]
    data_subset_agg = data_subset.groupby(['Treatment', 'Biological Replicate']).agg(
        Metabolic_Index_mean = ('Metabolic Index', 'mean'),
    ).reset_index()
    annotator = Annotator(ax, pairs, data=data_subset_agg,
                           x='Treatment', y='Metabolic_Index_mean')
    annotator.configure(test='t-test_welch', comparisons_correction='holm',
                            text_format='star', loc='inside', verbose=1, 
                            hide_non_significant=True, line_width=0.5)
    annotator.apply_test()
    ax, test_results = annotator.annotate()
    results = []
    for res in test_results:
        results.append(res.data.__dict__)
    results_df = pd.DataFrame(results)
    results_df['condition'] = condition
    all_results.append(results_df)
# Save all results to a CSV
all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(os.path.join(SAVE_DIR, 'Fig5_Metabolic_Index_statistical_results.csv'), index=False)


# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Fig5_Metabolic_Index_violin_plots_with_stats.png'), dpi=300, bbox_inches='tight')

In [ ]:
data_to_analyse['photon_counts_quartile'] = pd.qcut(data_to_analyse['Median Photon Counts'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

g=sns.FacetGrid(data_to_analyse, row='condition', row_order=['WT', 'DB'],
                height=3.5, aspect=2, sharex=True, despine=True)
g.map_dataframe(
    sns.violinplot, y='Metabolic Index', x='Treatment', hue='photon_counts_quartile', 
    palette='Set2', cut=0, dodge=True,
    inner='quartiles', alpha=1, color = 'gray', order=['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
).add_legend(title='Photon Counts Quartile', bbox_to_anchor=(0.8, 0.9), loc='upper left')
# g.map_dataframe(
#     sns.swarmplot, y='Metabolic Index', x='Treatment', 
#     hue='Biological Replicate', size=0.7, alpha=0.9, palette = 'Set2'
# ).add_legend(title='Biological Replicate', markerscale=10, bbox_to_anchor=(0.85, 0.5), loc='center left')
# g.map_dataframe(
#     sns.swarmplot, y='Metabolic Index', x='Treatment', hue='photon_counts_quartile', size=0.5, alpha=0.7, palette='Set2'
# ).add_legend(title='Photon Counts Quartile', markerscale=10, bbox_to_anchor=(0.8, 0.5), loc='center left')
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_xlabels('')
g.set_ylabels('Metabolic Index')
#Statistical annotation

pairs = [
    ('NA', 'AA'),
    ('NA', 'CA'),
    ('AA', 'CA'),
    ('NA', 'ES1'),
    ('NA', 'ES2'),
    ('NA', 'ES3'),
    ('NA', 'ES4'),
]

all_results = []
for ax, condition in zip(g.axes.flat, g.row_names):
    xticks = ax.get_xticks()
    for i, x in enumerate(xticks):
        if i % 2 == 0:
            ax.axvspan(
                x - 0.5,
                x + 0.5,
                color="0.95",
                zorder=0
            )
    ax.set_axisbelow(True)
#     data_subset = data_to_analyse[data_to_analyse['condition'] == condition]
#     data_subset_agg = data_subset.groupby(['Treatment', 'Biological Replicate']).agg(
#         Metabolic_Index_mean = ('Metabolic Index', 'mean'),
#     ).reset_index()
#     annotator = Annotator(ax, pairs, data=data_subset_agg,
#                            x='Treatment', y='Metabolic_Index_mean')
#     annotator.configure(test='t-test_welch', comparisons_correction='holm',
#                             text_format='star', loc='inside', verbose=1, 
#                             hide_non_significant=True, line_width=0.5)
#     annotator.apply_test()
#     ax, test_results = annotator.annotate()
#     results = []
#     for res in test_results:
#         results.append(res.data.__dict__)
#     results_df = pd.DataFrame(results)
#     results_df['condition'] = condition
#     all_results.append(results_df)
# # Save all results to a CSV
# all_results_df = pd.concat(all_results, ignore_index=True)
# all_results_df.to_csv(os.path.join(SAVE_DIR, 'Fig5_Metabolic_Index_statistical_results.csv'), index=False)


# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Fig6_Metabolic_Index_violin_plots_photon_counts.png'), dpi=300, bbox_inches='tight')

## Phase tau stats

In [ ]:
data_to_analyse['tau_calculated_ns'] = data_to_analyse['s_corrected']/(data_to_analyse['g_corrected']*OMEGA)*1e9  # in ns
# Aggregate by condition, biological replicate, and treatment to get median Metabolic Index and counts
stats_data = data_to_analyse.groupby(['condition', 'Biological Replicate', 'Treatment']).agg(
    tau = ('tau_calculated_ns', 'median'),
    cell_count = ('tau_calculated_ns', 'count')).reset_index()
# Save stats data
stats_data.to_csv(os.path.join(SAVE_DIR, f'Fig7_Stats_data.csv'), index=False)

In [ ]:
# Apply Friedman test to compare Metabolic Index across treatments within each condition
from scipy.stats import friedmanchisquare
treatments_to_compare = ['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
results = []
for condition in ['WT', 'DB']:
    condition_data = stats_data[(stats_data['condition'] == condition) &
                                 (stats_data['Treatment'].isin(treatments_to_compare))]
    # Pivot data to have treatments as columns
    stats_data_wide = condition_data.pivot_table(index=['condition', 'Biological Replicate'],
                                                columns='Treatment', values='tau')
    # Apply Friedman test
    stat, p = friedmanchisquare(*[stats_data_wide[col] for col in stats_data_wide.columns])
    # Store results
    results_df = pd.DataFrame({
        'condition': [condition],
        'friedman_statistic': [stat],
        'p_value': [p]
    })
    results.append(results_df)

    print(f"Friedman test for {condition}: stat={stat}, p={p}")
# Combine results
results_df = pd.concat(results, ignore_index=True)
results_df.to_csv(os.path.join(SAVE_DIR, f'Fig7_Friedman_test_results.csv'), index=False)

# Apply paired post-hoc Wilcoxon tests with Holm correction between treatments within each condition
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import itertools
results = []
for condition in ['WT', 'DB']:
    condition_data = stats_data[(stats_data['condition'] == condition) &
                                 (stats_data['Treatment'].isin(treatments_to_compare))]
    # Pivot data to have treatments as columns
    stats_data_wide = condition_data.pivot_table(index=['condition', 'Biological Replicate'],
                                                columns='Treatment', values='tau')
    # Perform pairwise Wilcoxon tests
    #treatment_pairs = list(itertools.combinations(treatments_to_compare, 2))
    # Only compare each AA/CA/ES treatment to NA control, plus AA vs CA
    treatment_pairs = [(treatments_to_compare[0], tr) for tr in treatments_to_compare[1:]] + [('AA', 'CA')]
    print(f"Post-hoc Wilcoxon tests for {condition}:")
    for pair in treatment_pairs:
        try:
            stat, p = wilcoxon(stats_data_wide[pair[0]], stats_data_wide[pair[1]])
            # Apply Holm correction later, but store uncorrected p-values for now
            # store pair, stat and p in df
            df_result = pd.DataFrame({'Condition': [condition],
                                      'Treatment 1': [pair[0]],
                                      'Treatment 2': [pair[1]],
                                      'Wilcoxon stat': [stat],
                                      'p-value': [p]})
            results.append(df_result)

            print(f"  {pair[0]} vs {pair[1]}: stat={stat}, p={p}")
        except ValueError as e:
            print(f"  {pair[0]} vs {pair[1]}: Could not perform test ({e})")
    # Apply Holm correction to p-values for this condition
    p_values = [r['p-value'].values[0] for r in results if r['Condition'].values[0] == condition]
    reject, pvals_corrected, _, _ = multipletests(p_values, method='holm')
    # Update results with corrected p-values
    idx = 0
    for i in range(len(results)):
        if results[i]['Condition'].values[0] == condition:
            results[i]['p-value (Holm corrected)'] = pvals_corrected[idx]
            results[i]['Reject Null'] = reject[idx]
            idx += 1
            
# Combine results into a single DataFrame
results_df = pd.concat(results, ignore_index=True)
results_df.to_csv(os.path.join(SAVE_DIR, f'Fig7_Wilcoxon_posthoc_results.csv'), index=False)

In [ ]:
g=sns.FacetGrid(data_to_analyse, row='condition', row_order=['WT', 'DB'],
                height=2.8, aspect=2, sharex=True, despine=True)
g.map_dataframe(
    sns.violinplot, y='tau_calculated_ns', x='Treatment', hue='Biological Replicate', 
    palette='Set2', cut=0, dodge=True,
    inner='quartiles', alpha=1, color = 'gray', order=['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
).add_legend(title='Biological\nReplicate', bbox_to_anchor=(0.8, 0.5), loc='center left')
# g.map_dataframe(
#     sns.swarmplot, y='Metabolic Index', x='Treatment', 
#     hue='Biological Replicate', size=0.7, alpha=0.9, palette = 'Set2'
# ).add_legend(title='Biological Replicate', markerscale=10, bbox_to_anchor=(0.85, 0.5), loc='center left')
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_xlabels('')
g.set_ylabels('Phase Lifetime (ns)\n' + r'$\tau_{\mathrm{\phi}}=\frac{S}{\omega G}$ ')
#Statistical annotation

pairs = [
    ('NA', 'AA'),
    ('NA', 'CA'),
    ('AA', 'CA'),
    ('NA', 'ES1'),
    ('NA', 'ES2'),
    ('NA', 'ES3'),
    ('NA', 'ES4'),
]

all_results = []
for ax, condition in zip(g.axes.flat, g.row_names):
    xticks = ax.get_xticks()
    for i, x in enumerate(xticks):
        if i % 2 == 0:
            ax.axvspan(
                x - 0.5,
                x + 0.5,
                color="0.95",
                zorder=0
            )
    ax.set_axisbelow(True)
    data_subset = data_to_analyse[data_to_analyse['condition'] == condition]
    data_subset_agg = data_subset.groupby(['Treatment', 'Biological Replicate']).agg(
        tau_mean = ('tau_calculated_ns', 'mean'),
    ).reset_index()
    annotator = Annotator(ax, pairs, data=data_subset_agg,
                           x='Treatment', y='tau_mean')
    annotator.configure(test='t-test_welch', comparisons_correction='holm',
                            text_format='star', loc='inside', verbose=1, 
                            hide_non_significant=True, line_width=0.5)
    annotator.apply_test()
    ax, test_results = annotator.annotate()
    results = []
    for res in test_results:
        results.append(res.data.__dict__)
    results_df = pd.DataFrame(results)
    results_df['condition'] = condition
    all_results.append(results_df)
# Save all results to a CSV
all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(os.path.join(SAVE_DIR, 'Fig7_Fast_Lifetime_statistical_results.csv'), index=False)


# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Fig7_Fast_Lifetime_violin_plots_with_stats.png'), dpi=300, bbox_inches='tight')

In [ ]:
# Aggregate by technical replicate then plot fast lifetime
techrep_data = data_to_analyse.groupby(['condition', 'Biological Replicate', 'Treatment', 'Technical Replicate']).agg(
    tau_mean = ('tau_calculated_ns', 'median'),
    counts = ('tau_calculated_ns', 'count')
).reset_index()
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)
g=sns.FacetGrid(techrep_data, row='condition', row_order=['WT', 'DB'],
                height=2.8, aspect=2.5, sharex=True, despine=True)
g.map_dataframe(
    sns.swarmplot, y='tau_mean', x='Treatment', hue='Biological Replicate', 
    palette='Set2', dodge=True, s=5, edgecolor='k', linewidth=0.5,
    #inner='quartiles', 
    alpha=1, color = 'gray', order=['NA', 'AA', 'CA', 'ES1', 'ES2', 'ES3', 'ES4']
).add_legend(title='Biological\nReplicate', bbox_to_anchor=(0.9, 0.5), loc='center left')
# g.map_dataframe(
#     sns.swarmplot, y='Metabolic Index', x='Treatment', 
#     hue='Biological Replicate', size=0.7, alpha=0.9, palette = 'Set2'
# ).add_legend(title='Biological Replicate', markerscale=10, bbox_to_anchor=(0.85, 0.5), loc='center left')
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_xlabels('')
g.set_ylabels('Fast Lifetime\n' + r'$\tau_{\mathrm{fast}}=\frac{G}{\omega S}$ (ns)')
#Statistical annotation

pairs = [
    ('NA', 'AA'),
    ('NA', 'CA'),
    ('AA', 'CA'),
    ('NA', 'ES1'),
    ('NA', 'ES2'),
    ('NA', 'ES3'),
    ('NA', 'ES4'),
]

all_results = []
for ax, condition in zip(g.axes.flat, g.row_names):
    xticks = ax.get_xticks()
    for i, x in enumerate(xticks):
        if i % 2 == 0:
            ax.axvspan(
                x - 0.5,
                x + 0.5,
                color="0.95",
                zorder=0
            )
    ax.set_axisbelow(True)
    data_subset = techrep_data[techrep_data['condition'] == condition]
    data_subset_agg = data_subset.groupby(['Treatment', 'Biological Replicate']).agg(
        tau_mean = ('tau_mean', 'mean'),
    ).reset_index()
    annotator = Annotator(ax, pairs, data=data_subset_agg,
                           x='Treatment', y='tau_mean')
    annotator.configure(test='t-test_welch', comparisons_correction='holm',
                            text_format='star', loc='inside', verbose=1, 
                            hide_non_significant=True, line_width=0.5)
    annotator.apply_test()
    ax, test_results = annotator.annotate()
    results = []
    for res in test_results:
        results.append(res.data.__dict__)
    results_df = pd.DataFrame(results)
    results_df['condition'] = condition
    all_results.append(results_df)
# Save all results to a CSV
all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(os.path.join(SAVE_DIR, 'Fig8_Fast_Lifetime_by_FOV_statistical_results.csv'), index=False)


# Save figure
plt.savefig(os.path.join(SAVE_DIR, 'Fig8_Fast_Lifetime_by_FOV_violin_plots_with_stats.png'), dpi=300, bbox_inches='tight')

## Friedman + paired Wilcoxon post-hoc + Benjamini-Krieger-Yekuteieli (BKY) FDR correction

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon

def bh_reject(pvals, q):
    """
    BH step-up rejection set (boolean mask in original order).
    """
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    order = np.argsort(pvals)
    p_sorted = pvals[order]
    thresh = (np.arange(1, m + 1) / m) * q
    passed = p_sorted <= thresh
    if not np.any(passed):
        return np.zeros(m, dtype=bool)
    k = np.max(np.where(passed)[0])  # index in sorted array
    reject_sorted = np.zeros(m, dtype=bool)
    reject_sorted[: k + 1] = True
    reject = np.zeros(m, dtype=bool)
    reject[order] = reject_sorted
    return reject

def bh_adjusted_pvals(pvals):
    """
    BH adjusted p-values ("q-values" under BH), in original order.
    """
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    order = np.argsort(pvals)
    p_sorted = pvals[order]
    adj_sorted = (m / np.arange(1, m + 1)) * p_sorted
    # enforce monotonicity
    adj_sorted = np.minimum.accumulate(adj_sorted[::-1])[::-1]
    adj_sorted = np.clip(adj_sorted, 0, 1)
    adj = np.empty(m, dtype=float)
    adj[order] = adj_sorted
    return adj

def bky_two_stage(pvals, q=0.05):
    """
    Two-stage step-up BKY FDR procedure (Benjamini, Krieger, Yekutieli).
    Returns:
      reject: boolean mask
      qvals: BKY-adjusted p-values (a practical 'adjusted p' output)
      q2: stage-2 BH level used

    Algorithm (two-stage linear step-up):
      Stage 1: BH at q' = q/(1+q). Let R1 = #rejections.
      Estimate m0 = m - R1. (If R1=0, no rejections.)
      Stage 2: BH at q* = q * m / m0 (capped at 1). :contentReference[oaicite:2]{index=2}
    """
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    if m == 0:
        return np.array([], dtype=bool), np.array([], dtype=float), np.nan

    q1 = q / (1.0 + q)
    rej1 = bh_reject(pvals, q1)
    R1 = int(rej1.sum())
    if R1 == 0:
        # No discoveries at stage 1 -> BKY rejects none
        # q-values: you can still report BH-adjusted p-values as a conservative summary
        return np.zeros(m, dtype=bool), bh_adjusted_pvals(pvals), q1

    m0_hat = max(m - R1, 1)  # avoid divide-by-zero
    q2 = min(q * m / m0_hat, 1.0)
    reject = bh_reject(pvals, q2)

    # Practical adjusted p-values:
    # Use BH-adjusted p-values scaled by (m0_hat/m), then cap to 1.
    # This aligns with the stage-2 BH level interpretation.
    bh_q = bh_adjusted_pvals(pvals)
    qvals = np.clip((m0_hat / m) * bh_q, 0, 1)

    return reject, qvals, q2


In [ ]:

# -----------------------------
# Example end-to-end workflow
# -----------------------------
def friedman_with_bky_posthoc(df, subject_col, treatment_col, value_col, q=0.05):
    """
    df must be one row per subject (mouse) x treatment (i.e., already aggregated),
    with all treatments present per subject (balanced).
    """
    wide = df.pivot(index=subject_col, columns=treatment_col, values=value_col)
    # drop subjects with any missing condition (Friedman needs complete blocks)
    wide = wide.dropna(axis=0, how="any")

    treatments = list(wide.columns)

    # Friedman omnibus
    stat, p_fried = friedmanchisquare(*[wide[t] for t in treatments])

    # Paired Wilcoxon post-hoc (all pairs)
    #pairs = list(itertools.combinations(treatments, 2))
    pairs = [('NA', tr) for tr in treatments[:-1]] + [('AA', 'CA')]
    rows = []
    for a, b in pairs:
        w_stat, p = wilcoxon(wide[a], wide[b])
        rows.append({"comparison": f"{a} vs {b}", "p_raw": p, "W": w_stat})
    posthoc = pd.DataFrame(rows)

    # BKY correction across the pairwise tests
    reject, qvals, q2 = bky_two_stage(posthoc["p_raw"].to_numpy(), q=q)
    posthoc["p_bky"] = qvals
    posthoc["reject_bky"] = reject

    return {
        "friedman_stat": stat,
        "friedman_p": p_fried,
        "bky_stage2_level_q2": q2,
        "posthoc": posthoc.sort_values("p_raw")
    }


In [ ]:
stats_results = []
for condition in ['WT', 'DB']:
    condition_data = stats_data[(stats_data['condition'] == condition) &
                                 (stats_data['Treatment'].isin(treatments_to_compare))]
    result = friedman_with_bky_posthoc(
        condition_data,
        subject_col='Biological Replicate',
        treatment_col='Treatment',
        value_col='tau',
        q=0.05
    )
    print(f"Condition: {condition}")
    print(f"Friedman statistic: {result['friedman_stat']}, p-value: {result['friedman_p']}")
    print(f"BKY stage-2 level q2: {result['bky_stage2_level_q2']}")
    print("Post-hoc results:")
    print(result['posthoc'])
    # Store results
    posthoc_df = result['posthoc']
    posthoc_df['condition'] = condition
    stats_results.append(posthoc_df)
# Combine all results
all_stats_results = pd.concat(stats_results, ignore_index=True)
all_stats_results.to_csv(os.path.join(SAVE_DIR, f'Fig7_BKY_posthoc_results.csv'), index=False)